# Model Tuning (Random Dev + OOT)
## Loan-default Probability-of-Default (PD) model

Same pipeline as `02_model_tuning.ipynb` with **Train–Val–Test–OOT**: random stratified dev splits + temporal OOT.
Exports to `model_store/best_params_random_dev.json`.


In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make the repo importable regardless of where the kernel starts.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, f1_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, PredefinedSplit
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance

from utils.config import (
    COST_FALSE_NEGATIVE, COST_FALSE_POSITIVE, MODEL_STORE_DIR,
    BEST_PARAMS_FILENAME, CHAMPION_CONFIG_FILENAME,
)
from utils.ml.dataset import (
    load_training_frame, select_feature_columns, coerce_feature_matrix,
    random_dev_oot_split, ks_statistic,
)
from utils.ml.train import _build_candidates, _evaluate, _best_cost_threshold

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("Repo root:", ROOT)

## 1. Load the point-in-time training table

`g2_training_table` is the **leakage-safe** modelling table from the medallion pipeline. Each row is one loan joined to the **exact `g2_feature_store` row that inference scores** — the feature snapshot **as-of the application month** — so `feature_snapshot_date = loan_start_date < label_snapshot_date`. The label is `30dpd@mob6`, observed 6 months after disbursement. (Anchoring features to the *application* date — not the label-maturity date — is critical: a model that decides at application must not see any post-disbursement behaviour.)

This design defends against all three leakage types called out in the brief:
- **Target leakage** — only application-time features are joined; no repayment/outcome fields leak in.
- **Temporal leakage** — features anchored to the application month + OOT validation by **application cohort** (Section 3); train/val/test are random. No train/serve skew: training and inference read the same feature store.
- **Train/test contamination** — imputation & scaling are fit *inside* the cross-validation pipeline, never on the full dataset.

In [ ]:
# `as_of` = latest available label snapshot. load_training_frame keeps rows whose
# labels are observable by this date and renames `label` -> `default_flag`.
AS_OF = "2024-12-01"
df = load_training_frame(AS_OF)

feature_cols = select_feature_columns(df)
df["loan_start_date"] = pd.to_datetime(df["loan_start_date"])      # application date
df["label_snapshot_date"] = pd.to_datetime(df["label_snapshot_date"])  # +mob maturity
df["feature_snapshot_date"] = pd.to_datetime(df["feature_snapshot_date"])

# Leakage guard: features must be known at application time, never after.
assert (df["feature_snapshot_date"] <= df["loan_start_date"]).all(), "feature leakage detected!"

print(f"Rows (labelled loans)  : {len(df):,}")
print(f"Model features         : {len(feature_cols)}")
print(f"Application cohorts     : {df['loan_start_date'].nunique()} "
      f"({df['loan_start_date'].min():%Y-%m} -> {df['loan_start_date'].max():%Y-%m})")
print(f"Overall bad rate        : {df['default_flag'].mean():.3f}")
print("Feature anchor == application month for all rows:",
      bool((df['feature_snapshot_date'] == df['loan_start_date']).all()))
df[["loan_id", "loan_start_date", "feature_snapshot_date", "label_snapshot_date", "default_flag"]].head()

## 2. Data volume & class balance → model-family decision

| Property | Value | Implication |
|---|---|---|
| Labelled rows | ~9k | Mid-size tabular. Enough for trees/GBMs; **too small for deep learning** (no benefit on structured data at this scale). |
| Features | ~112 | Wide, mixed numeric + one-hot. Tree ensembles handle interactions & monotone-ish effects with no manual feature crossing. |
| Bad rate | ~29% | Moderate imbalance — **not** rare-event. AUC/PR are both usable; we still weight classes. |
| Cohort bad rate | stable 26–32% | Stable target prior → a single trained champion generalises across months (validated in monitoring). |

**Decision.** Benchmark three families that span the interpretability/performance frontier and are the *de-facto industry standard for application credit scoring*:
1. **Logistic Regression** — the regulator-friendly **scorecard** baseline (transparent, monotonic, easy to explain to credit officers and auditors).
2. **Histogram Gradient Boosting** — sklearn's in-process GBM workhorse; native NaN handling, strong out-of-the-box, ships in production.
3. **LightGBM** — Microsoft GBM challenger (same family as XGBoost/HGB); included in the bake-off when `lightgbm` is installed locally (`pip install -r requirements-dev.txt`). Tuned separately in Section 5a.

**Primary metric: ROC-AUC / Gini (`Gini = 2·AUC − 1`).** This is the standard discrimination metric for application scorecards: threshold-independent and stable under the moderate class imbalance. We report **KS** (regulator favourite), **PR-AUC** (imbalance-aware), and **Brier score** (calibration quality) as supporting metrics, because a PD that feeds expected-loss and pricing must also be *well-calibrated*, not just well-ranked.

In [ ]:
# Cohort volume & bad-rate stability by APPLICATION month — the monitoring baseline.
cohort = (df.groupby(df["loan_start_date"].dt.to_period("M"))
            .agg(n=("default_flag", "size"), bad_rate=("default_flag", "mean")))
cohort.index = cohort.index.astype(str)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
cohort["n"].plot(kind="bar", ax=ax[0], color="#4C72B0")
ax[0].set_title("Labelled loans per application cohort"); ax[0].set_ylabel("# loans"); ax[0].set_xlabel("")
cohort["bad_rate"].plot(marker="o", ax=ax[1], color="#C44E52")
ax[1].axhline(df["default_flag"].mean(), ls="--", color="grey", label="overall")
ax[1].set_title("Bad rate (30dpd@mob6) by application cohort"); ax[1].set_ylabel("bad rate"); ax[1].set_ylim(0, 0.45)
ax[1].legend(); plt.tight_layout(); plt.show()
cohort.round(3)

## 3. Train–Val–Test–OOT split

1. **OOT (temporal)** — last **3** application cohorts, sealed until the final headline score.
2. **Train / Val / Test (random)** — stratified 70 / 15 / 15 on all older cohorts.

### Do we need cross-validation?

At ~7.5k development rows with dedicated **val** (~1.1k), **test** (~1.1k), and **OOT** (~1.5k) holdouts, **k-fold CV is not required**. One validation fold is enough to rank hyper-parameters; the random test checks in-sample generalisation; OOT is the temporal truth.

Section 5 uses a **predefined train→val split** for `RandomizedSearchCV` (not k-fold).

In [ ]:
OOT_COHORTS = 3
TEST_FRAC = 0.15
VAL_FRAC = 0.15

train_df, val_df, test_df, oot_df = random_dev_oot_split(
    df,
    n_oot_cohorts=OOT_COHORTS,
    test_size=TEST_FRAC,
    val_size=VAL_FRAC,
    random_state=RANDOM_STATE,
)

def xy(frame):
    return coerce_feature_matrix(frame, feature_cols), frame["default_flag"].astype(int)

X_train, y_train = xy(train_df)
X_val,   y_val   = xy(val_df)
X_test,  y_test  = xy(test_df)
X_oot,   y_oot   = xy(oot_df)

X_dev = pd.concat([X_train, X_val]); y_dev = pd.concat([y_train, y_val])

for name, frame in [("train", train_df), ("val", val_df), ("test", test_df), ("oot", oot_df)]:
    cohorts = sorted(frame["loan_start_date"].dt.strftime("%Y-%m").unique())
    print(
        f"{name:>5}: n={len(frame):5d}  bad_rate={frame['default_flag'].mean():.3f}  "
        f"cohorts={cohorts[0]}..{cohorts[-1]} ({len(cohorts)} months)"
    )

## 4. Candidate bake-off (default hyper-parameters)

Each candidate is a full sklearn `Pipeline` (median-impute → scale (linear only) → model). Fitting the imputer/scaler *inside* the pipeline guarantees they are learned on the training fold only — no train/test contamination. We fit on `train`, score on `validation`, and compare on the full metric panel.

In [ ]:
candidates = _build_candidates(feature_cols, tuned={})
print("Bake-off families:", ", ".join(candidates))
if "lightgbm" not in candidates:
    print("  (install LightGBM: pip install -r requirements-dev.txt)")
rows = []
fitted = {}
for name, pipe in candidates.items():
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    tr = _evaluate(name, y_train, pipe.predict_proba(X_train)[:, 1])  # in-sample
    va = _evaluate(name, y_val, pipe.predict_proba(X_val)[:, 1])      # out-of-sample
    rows.append({
        "model": name,
        "train_auc": tr["roc_auc"], "val_auc": va["roc_auc"],
        "auc_gap": tr["roc_auc"] - va["roc_auc"],   # overfit signal (large +ve = overfitting)
        "val_gini": va["gini"], "val_ks": va["ks"],
        "val_pr_auc": va["pr_auc"], "val_brier": va["brier"],
    })

baseline_metrics = (pd.DataFrame(rows).set_index("model")
                      .sort_values("val_auc", ascending=False))
# `auc_gap` is the train-minus-validation AUC: ~0 = well-fit; large +ve = overfitting;
# both train & val low = underfitting. Trees are capped (depth/leaf/min_samples) precisely
# to keep this gap small at this data volume.
baseline_metrics.round(4)

## 5. Hyper-parameter tuning (champion family)

Tune the bake-off winner with `RandomizedSearchCV` on a **single train→val split** (no k-fold CV). Random **test** is evaluated after tuning; **OOT** stays sealed until Section 6.


In [ ]:
champion = baseline_metrics.index[0]
lgbm_best_params = None  # populated in Section 5a when LightGBM is tuned
print("Champion family from bake-off:", champion)

# ---- Predefined train→val split for tuning (no k-fold CV) ----
X_tune = pd.concat([X_train, X_val])
y_tune = pd.concat([y_train, y_val])
_test_fold = np.full(len(X_tune), -1)
_test_fold[len(X_train):] = 0
cv = PredefinedSplit(test_fold=_test_fold)
print(f"Predefined tuning split: train n={len(X_train)}, val n={len(X_val)} (no k-fold CV)")

# Search spaces are keyed on the pipeline's `model__` step.
search_spaces = {
    "hist_gradient_boosting": {
        "model__learning_rate": [0.02, 0.05, 0.1],
        "model__max_iter": [200, 400, 600],
        "model__max_depth": [2, 3, 4, None],
        "model__max_leaf_nodes": [15, 31, 63],
        "model__l2_regularization": [0.0, 1.0, 5.0],
        "model__min_samples_leaf": [20, 50, 100],
    },
    "logistic_regression": {
        "model__C": np.logspace(-2, 1, 10),
        "model__penalty": ["l2"],
    },
    "lightgbm": {
        "model__learning_rate": [0.02, 0.05, 0.1],
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [2, 3, 4, -1],
        "model__num_leaves": [15, 31, 63],
        "model__min_child_samples": [20, 50, 100],
        "model__reg_lambda": [0.0, 1.0, 5.0],
        "model__subsample": [0.7, 0.8, 1.0],
        "model__colsample_bytree": [0.7, 0.8, 1.0],
    },
}

from sklearn.base import clone

search = RandomizedSearchCV(
    _build_candidates(feature_cols, tuned={})[champion],
    param_distributions=search_spaces[champion],
    n_iter=200 if champion == "lightgbm" else 20, scoring="roc_auc", cv=cv, n_jobs=-1,
    random_state=RANDOM_STATE, refit=True, verbose=0,
)
search.fit(X_tune, y_tune)

# Per-trial bake-off panel (same metrics as Section 4) on the random train/val split,
# plus the validation ROC-AUC each trial was scored on during search.
trial_rows = []
for i, params in enumerate(search.cv_results_["params"]):
    pipe = clone(search.estimator).set_params(**params)
    pipe.fit(X_train, y_train)
    tr = _evaluate(champion, y_train, pipe.predict_proba(X_train)[:, 1])
    va = _evaluate(champion, y_val, pipe.predict_proba(X_val)[:, 1])
    row = {
        "trial": i + 1,
        "cv_auc": search.cv_results_["mean_test_score"][i],
        "train_auc": tr["roc_auc"],
        "val_auc": va["roc_auc"],
        "auc_gap": tr["roc_auc"] - va["roc_auc"],
        "val_gini": va["gini"],
        "val_ks": va["ks"],
        "val_pr_auc": va["pr_auc"],
        "val_brier": va["brier"],
    }
    row.update({k.replace("model__", ""): v for k, v in params.items()})
    trial_rows.append(row)

_metric_cols = [
    "trial", "cv_auc", "train_auc", "val_auc", "auc_gap",
    "val_gini", "val_ks", "val_pr_auc", "val_brier",
]
_param_cols = sorted(c for c in trial_rows[0] if c not in _metric_cols)
tuning_metrics = (
    pd.DataFrame(trial_rows)[_metric_cols + _param_cols]
    .sort_values("cv_auc", ascending=False)
    .reset_index(drop=True)
)

best_params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}

# Prefer best CV AUC among trials with acceptable train-val gap (LightGBM overfits easily).
AUC_GAP_CAP = 0.05
_feasible = tuning_metrics[tuning_metrics["auc_gap"] <= AUC_GAP_CAP].sort_values("cv_auc", ascending=False)
if not _feasible.empty:
    _pick = _feasible.iloc[0]
    _i = int(_pick["trial"]) - 1
    best_params = {k.replace("model__", ""): v for k, v in search.cv_results_["params"][_i].items()}
    search.best_estimator_ = clone(search.estimator).set_params(**search.cv_results_["params"][_i])
    search.best_estimator_.fit(X_dev, y_dev)
    print(
        f"\nGap-aware pick: trial {int(_pick['trial'])} "
        f"(cv_auc={_pick['cv_auc']:.4f}, auc_gap={_pick['auc_gap']:.4f}, gap cap={AUC_GAP_CAP})"
    )
elif champion == "lightgbm":
    print(
        f"\nNo trial with auc_gap <= {AUC_GAP_CAP}; still using best CV AUC. "
        "Try tightening the search space further or raising n_iter."
    )


In [ ]:
MANUAL_TRIAL = 159
if MANUAL_TRIAL is not None:
    _i = int(MANUAL_TRIAL) - 1
    best_params = {k.replace("model__", ""): v for k, v in search.cv_results_["params"][_i].items()}
    search.best_estimator_ = clone(search.estimator).set_params(**search.cv_results_["params"][_i])
    search.best_estimator_.fit(X_dev, y_dev)
    _row = tuning_metrics.loc[tuning_metrics["trial"] == MANUAL_TRIAL].iloc[0]
    print(
        f"\nManual pick: trial {MANUAL_TRIAL} "
        f"(cv_auc={_row['cv_auc']:.4f}, auc_gap={_row['auc_gap']:.4f})"
    )
    print("Best params:", {k.replace("model__", ""): v for k, v in best_params.items()})

bl = baseline_metrics.loc[champion]
print(
    f"Section 4 baseline ({champion}, defaults): "
    f"train_auc={bl['train_auc']:.4f}  val_auc={bl['val_auc']:.4f}  "
    f"auc_gap={bl['auc_gap']:.4f}  val_gini={bl['val_gini']:.4f}"
)
print(f"Best val ROC-AUC: {search.best_score_:.4f}  |  Best params: {best_params}")
print(f"\n{search.n_iter} tuning trials (sorted by val ROC-AUC; auc_gap = train_auc - val_auc):")
tuning_metrics.round(4)

### 5a. LightGBM hyper-parameter tuning (only when LightGBM did **not** win the bake-off)

Section 5 already tunes the bake-off **champion**. Run this cell only when the champion is *not* LightGBM (e.g. HGB won Section 4) and you still want tuned LightGBM params exported alongside the champion. If LightGBM won, this cell is skipped automatically.

In [ ]:
# print( baseline_metrics.index[1])

In [ ]:
# champion = baseline_metrics.index[1]
# lgbm_best_params = None  # populated in Section 5a when LightGBM is tuned
# # print("Champion family from bake-off:", champion)

# # ---- Temporal (out-of-time) cross-validation for tuning ----
# # Group dev rows by label cohort and use an EXPANDING window: each fold trains on all
# # earlier cohorts and validates on the next one. This keeps hyper-parameter selection
# # consistent with deployment (no future cohort leaks into a training fold), unlike a
# # random StratifiedKFold that would mix cohorts across folds. With a stable target
# # prior (bad rate 26-32% across cohorts) the practical gap to StratifiedKFold is small,
# # but temporal CV removes the last bit of optimism and matches the OOT philosophy.
# dev_df = pd.concat([train_df, val_df])
# dev_groups = dev_df["loan_start_date"].values  # application-cohort folds

# def expanding_cohort_cv(groups, y, min_train_cohorts=6):
#     uniq = np.array(sorted(pd.unique(groups)))
#     pos = np.arange(len(groups))
#     for i in range(min_train_cohorts, len(uniq)):
#         tr = pos[np.isin(groups, uniq[:i])]
#         va = pos[np.isin(groups, [uniq[i]])]
#         if len(va) and len(np.unique(np.asarray(y)[va])) > 1:
#             yield tr, va

# cv = list(expanding_cohort_cv(dev_groups, y_dev.values))
# print(f"Temporal CV: {len(cv)} expanding-window folds over {dev_df['label_snapshot_date'].nunique()} dev cohorts")

# # Search spaces are keyed on the pipeline's `model__` step.
# search_spaces = {
#     "hist_gradient_boosting": {
#         "model__learning_rate": [0.02, 0.05, 0.1],
#         "model__max_iter": [200, 400, 600],
#         "model__max_depth": [2, 3, 4, None],
#         "model__max_leaf_nodes": [15, 31, 63],
#         "model__l2_regularization": [0.0, 1.0, 5.0],
#         "model__min_samples_leaf": [20, 50, 100],
#     },
#     "logistic_regression": {
#         "model__C": np.logspace(-2, 1, 10),
#         "model__penalty": ["l2"],
#     },
#     "lightgbm": {
#         "model__learning_rate": [0.02, 0.05, 0.1],
#         "model__n_estimators": [200, 400, 600],
#         "model__max_depth": [2, 3, 4, -1],
#         "model__num_leaves": [15, 31, 63],
#         "model__min_child_samples": [20, 50, 100],
#         "model__reg_lambda": [0.0, 1.0, 5.0],
#         "model__subsample": [0.7, 0.8, 1.0],
#         "model__colsample_bytree": [0.7, 0.8, 1.0],
#     },
# }

# from sklearn.base import clone

# search = RandomizedSearchCV(
#     _build_candidates(feature_cols, tuned={})[champion],
#     param_distributions=search_spaces[champion],
#     n_iter=20, scoring="roc_auc", cv=cv, n_jobs=-1,
#     random_state=RANDOM_STATE, refit=True, verbose=0,
# )
# search.fit(X_dev, y_dev)

# # Per-trial bake-off panel (same metrics as Section 4) on the temporal train/val split,
# # plus the temporal-val ROC-AUC each trial was scored on during search.
# trial_rows = []
# for i, params in enumerate(search.cv_results_["params"]):
#     pipe = clone(search.estimator).set_params(**params)
#     pipe.fit(X_train, y_train)
#     tr = _evaluate(champion, y_train, pipe.predict_proba(X_train)[:, 1])
#     va = _evaluate(champion, y_val, pipe.predict_proba(X_val)[:, 1])
#     row = {
#         "trial": i + 1,
#         "cv_auc": search.cv_results_["mean_test_score"][i],
#         "train_auc": tr["roc_auc"],
#         "val_auc": va["roc_auc"],
#         "auc_gap": tr["roc_auc"] - va["roc_auc"],
#         "val_gini": va["gini"],
#         "val_ks": va["ks"],
#         "val_pr_auc": va["pr_auc"],
#         "val_brier": va["brier"],
#     }
#     row.update({k.replace("model__", ""): v for k, v in params.items()})
#     trial_rows.append(row)

# _metric_cols = [
#     "trial", "cv_auc", "train_auc", "val_auc", "auc_gap",
#     "val_gini", "val_ks", "val_pr_auc", "val_brier",
# ]
# _param_cols = sorted(c for c in trial_rows[0] if c not in _metric_cols)
# tuning_metrics = (
#     pd.DataFrame(trial_rows)[_metric_cols + _param_cols]
#     .sort_values("cv_auc", ascending=False)
#     .reset_index(drop=True)
# )

# best_params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}

# bl = baseline_metrics.loc[champion]
# print(
#     f"Section 4 baseline ({champion}, defaults): "
#     f"train_auc={bl['train_auc']:.4f}  val_auc={bl['val_auc']:.4f}  "
#     f"auc_gap={bl['auc_gap']:.4f}  val_gini={bl['val_gini']:.4f}"
# )
# print(f"Best val ROC-AUC: {search.best_score_:.4f}  |  Best params: {best_params}")
# print(f"\n{search.n_iter} tuning trials (sorted by val ROC-AUC; auc_gap = train_auc - val_auc):")
# tuning_metrics.round(4)

In [ ]:
# Export tuned params -> consumed automatically by utils/ml/train.py at next retrain.
def _to_jsonable(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    return v

best_params_clean = {k: _to_jsonable(v) for k, v in best_params.items()}
export = {champion: best_params_clean}
if lgbm_best_params:
    export["lightgbm"] = {k: _to_jsonable(v) for k, v in lgbm_best_params.items()}
out_path = MODEL_STORE_DIR / BEST_PARAMS_FILENAME
out_path.write_text(json.dumps(export, indent=2), encoding="utf-8")
print(f"Wrote tuned params -> {out_path}\n{json.dumps(export, indent=2)}")

In [ ]:
# # Export tuned params -> consumed automatically by utils/ml/train.py at next retrain.
# def _to_jsonable(v):
#     if isinstance(v, (np.integer,)):
#         return int(v)
#     if isinstance(v, (np.floating,)):
#         return float(v)
#     return v

# best_params_clean = {k: _to_jsonable(v) for k, v in best_params.items()}
# export = {champion: best_params_clean}
# if lgbm_best_params:
#     export["lightgbm"] = {k: _to_jsonable(v) for k, v in lgbm_best_params.items()}
# out_path = MODEL_STORE_DIR / BEST_PARAMS_FILENAME
# out_path.write_text(json.dumps(export, indent=2), encoding="utf-8")
# print(f"Wrote tuned params -> {out_path}\n{json.dumps(export, indent=2)}")

In [ ]:
## Higtogram GB
# # Interim holdout scores (full-feature tuned model — superseded by Section 11 reduced champion).
# # Tuned champion from Section 5 (RandomizedSearchCV + gap-aware pick).
# tuned_pipe = search.best_estimator_

# # Random test-fold check (in-sample holdout).
# test_score = tuned_pipe.predict_proba(X_test)[:, 1]
# holdout = _evaluate(f"{champion}_tuned", y_test, test_score)
# print("Random test-fold performance:")
# for k in ["roc_auc", "gini", "ks", "pr_auc", "brier"]:
#     print(f"  {k:>8}: {holdout[k]:.4f}")
# print()

# # Headline: tuned champion evaluated ONCE on the sealed OOT test cohorts.
# oot_score = tuned_pipe.predict_proba(X_oot)[:, 1]
# oot = _evaluate(f"{champion}_tuned", y_oot, oot_score)
# print("Out-of-time test performance (the number we trust):")
# for k in ["roc_auc", "gini", "ks", "pr_auc", "brier"]:
#     print(f"  {k:>8}: {oot[k]:.4f}")

In [ ]:
# Interim holdout scores (full-feature tuned model — superseded by Section 11 reduced champion).
# Tuned champion from Section 5 (RandomizedSearchCV + gap-aware pick).
tuned_pipe = search.best_estimator_

# Random test-fold check (in-sample holdout).
test_score = tuned_pipe.predict_proba(X_test)[:, 1]
holdout = _evaluate(f"{champion}_tuned", y_test, test_score)
print("Random test-fold performance:")
for k in ["roc_auc", "gini", "ks", "pr_auc", "brier"]:
    print(f"  {k:>8}: {holdout[k]:.4f}")
print()

# Headline: tuned champion evaluated ONCE on the sealed OOT test cohorts.
oot_score = tuned_pipe.predict_proba(X_oot)[:, 1]
oot = _evaluate(f"{champion}_tuned", y_oot, oot_score)
print("Out-of-time test performance (the number we trust):")
for k in ["roc_auc", "gini", "ks", "pr_auc", "brier"]:
    print(f"  {k:>8}: {oot[k]:.4f}")

### 5b. (Optional) XGBoost + Optuna

Heavier libraries are intentionally **not** in the production image (to keep the Docker build fast and reproducible). This cell upgrades the search to XGBoost via Optuna *if those packages are installed locally* (`pip install -r requirements-dev.txt`); otherwise it is skipped cleanly. If XGBoost wins materially, wire it into `utils/ml/train._build_candidates` before promoting.

In [ ]:
try:
    import optuna
    from xgboost import XGBClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.compose import ColumnTransformer

    scale_pos = float((y_dev == 0).sum() / max((y_dev == 1).sum(), 1))
    pre = ColumnTransformer([("num", SimpleImputer(strategy="median"), feature_cols)])

    def objective(trial):
        params = dict(
            n_estimators=trial.suggest_int("n_estimators", 200, 800, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 6),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            reg_lambda=trial.suggest_float("reg_lambda", 0.0, 5.0),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 20),
        )
        model = Pipeline([("pre", pre),
                          ("model", XGBClassifier(eval_metric="logloss",
                                                  scale_pos_weight=scale_pos,
                                                  random_state=RANDOM_STATE, **params))])
        from sklearn.model_selection import cross_val_score
        return cross_val_score(model, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE), scoring="roc_auc", n_jobs=-1).mean()

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study.optimize(objective, n_trials=30, show_progress_bar=False)
    print(f"Optuna best CV ROC-AUC: {study.best_value:.4f}")
    print("Optuna best params:", study.best_params)
except ImportError:
    print("optuna/xgboost not installed - skipping (pip install -r requirements-dev.txt to enable).")

## 6. Probability calibration

A ranking model is not enough: the output must be a **true PD** so it can feed expected loss (`EL = PD × LGD × EAD`), risk-based pricing and IFRS-9 provisioning. We wrap the tuned champion in `CalibratedClassifierCV` (isotonic) and compare reliability curves and Brier scores before/after. A well-calibrated curve hugs the diagonal.

> **Note:** Calibration below uses the **interim full-feature** tuned model from Section 5. The promoted champion is the feature-reduced model in Sections 11–12.


In [ ]:
calibrated = CalibratedClassifierCV(search.best_estimator_, method="isotonic", cv=3)
calibrated.fit(X_dev, y_dev)

raw_p = tuned_pipe.predict_proba(X_test)[:, 1]
cal_p = calibrated.predict_proba(X_test)[:, 1]

plt.figure(figsize=(6, 6))
for label, p in [("uncalibrated", raw_p), ("isotonic", cal_p)]:
    frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
    plt.plot(mean_pred, frac_pos, marker="o", label=f"{label} (Brier={brier_score_loss(y_test, p):.3f})")
plt.plot([0, 1], [0, 1], "--", color="grey", label="perfectly calibrated")
plt.xlabel("Mean predicted PD"); plt.ylabel("Observed default rate")
plt.title("Reliability curve (random test fold)"); plt.legend(); plt.tight_layout(); plt.show()

## 7. Decision threshold from business cost

The score is a probability; the *business* needs a cut-off. The naive 0.5 cut ignores the cost asymmetry. We choose the threshold that **minimises expected cost** on validation, weighting a missed default (false negative) `5×` a wrongful decline (false positive) — a placeholder for `LGD×EAD ÷ margin`, to be calibrated with Finance.

We also plot the operational trade-off the business actually cares about: **approval rate** vs **bad-rate among approved**. This is the slide that turns AUC into money.

In [ ]:
val_p = calibrated.predict_proba(X_val)[:, 1]
threshold, exp_cost = _best_cost_threshold(y_val, val_p)
print(f"Cost-optimal threshold = {threshold:.3f}  (FN:FP = {COST_FALSE_NEGATIVE}:{COST_FALSE_POSITIVE})")

grid = np.linspace(0.02, 0.6, 120)
appr, bad_appr, costs = [], [], []
for t in grid:
    approve = cal_p < t                      # approve when PD below threshold
    appr.append(approve.mean())
    bad_appr.append(y_test[approve].mean() if approve.any() else np.nan)
    fn = int(((cal_p < t) & (y_test == 1)).sum()); fp = int(((cal_p >= t) & (y_test == 0)).sum())
    costs.append(COST_FALSE_NEGATIVE * fn + COST_FALSE_POSITIVE * fp)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(grid, costs, color="#8172B3"); ax[0].axvline(threshold, ls="--", color="red")
ax[0].set_title("Expected business cost vs threshold (random test)"); ax[0].set_xlabel("PD threshold"); ax[0].set_ylabel("weighted cost")
ax[1].plot(appr, bad_appr, color="#4C72B0", marker=".")
ax[1].axhline(y_test.mean(), ls="--", color="grey", label=f"portfolio bad rate ({y_test.mean():.2f})")
ax[1].set_title("Approval rate vs bad-rate among approved (random test)"); ax[1].set_xlabel("approval rate"); ax[1].set_ylabel("bad rate among approved")
ax[1].legend(); plt.tight_layout(); plt.show()

approve = cal_p < threshold
print(f"At chosen threshold -> approval rate={approve.mean():.1%}, "
      f"bad-rate among approved={y_test[approve].mean():.1%} (vs {y_test.mean():.1%} portfolio)")

## 8. Interpretability — what drives the score?

Credit models must be explainable for governance and adverse-action reasons. **Permutation importance on validation** ranks features by the AUC drop when each is shuffled — a model-agnostic sanity check that never touches the sealed OOT test. **SHAP** (next cell) gives complementary global attributions on the same validation fold. Feature shortlisting for Section 10 is also derived from validation SHAP only; OOT stays blind until the final reduced-model score.

In [ ]:
perm = permutation_importance(
    tuned_pipe, X_val, y_val, scoring="roc_auc",
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1,
)
imp = (pd.Series(perm.importances_mean, index=feature_cols)
         .sort_values(ascending=False).head(20))
plt.figure(figsize=(8, 6))
imp.iloc[::-1].plot(kind="barh", color="#55A868")
plt.title("Top 20 features by permutation importance (AUC drop, validation)")
plt.xlabel("mean AUC decrease"); plt.tight_layout(); plt.show()
imp.round(4)

In [ ]:
# SHAP global summary on validation (same fold used for feature shortlisting in Section 10).
try:
    import shap

    _pre = tuned_pipe.named_steps["preprocessor"]
    _model = tuned_pipe.named_steps["model"]
    X_val_t = _pre.transform(X_val)
    explainer = shap.TreeExplainer(_model)
    sample = shap.utils.sample(X_val_t, min(500, len(X_val_t)), random_state=RANDOM_STATE)
    shap_values = explainer.shap_values(sample)
    if isinstance(shap_values, list):
        shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]

    # Top 10 by mean |SHAP| — compact bar chart for slides.
    top10 = (
        pd.Series(np.abs(shap_values).mean(axis=0), index=feature_cols)
        .sort_values(ascending=False)
        .head(10)
    )

    fig, ax = plt.subplots(figsize=(4.5, 3.0))
    y_pos = np.arange(len(top10))
    ax.barh(
        y_pos,
        top10.values[::-1],
        height=0.62,
        color="#4C72B0",
        edgecolor="white",
        linewidth=0.6,
    )
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top10.index[::-1], fontsize=7.5)
    ax.set_xlabel("Mean |SHAP|", fontsize=8.5, labelpad=4)
    ax.set_title("Top 10 SHAP feature importance", fontsize=9.5, fontweight="600", pad=6)
    ax.tick_params(axis="x", labelsize=8, length=3, width=0.6)
    ax.tick_params(axis="y", length=0)
    ax.grid(False)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_color("#bbbbbb")
    ax.set_xlim(0, top10.max() * 1.08)
    fig.subplots_adjust(left=0.38, right=0.97, top=0.88, bottom=0.14)
    plt.show()
except Exception as exc:
    print(f"SHAP skipped ({type(exc).__name__}): install `shap` and use a tree model to enable.")

## 9. Fit diagnostics — over/under-fitting (learning curve)

The bake-off table already reports the **train − validation AUC gap** for every family (a small gap ⇒ well-fit; large positive ⇒ overfit; both low ⇒ underfit). Here we go one level deeper for the tuned champion with a **learning curve**: train vs cross-validated AUC as the training set grows. Converging curves with a small final gap confirm the model generalises; a wide, non-closing gap would tell us to regularise harder or add data.

In [ ]:
from sklearn.model_selection import learning_curve

sizes, train_sc, val_sc = learning_curve(
    search.best_estimator_, X_dev, y_dev, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE), scoring="roc_auc",
    train_sizes=np.linspace(0.3, 1.0, 6), n_jobs=-1, shuffle=True, random_state=RANDOM_STATE,
)
tr_m, tr_s = train_sc.mean(1), train_sc.std(1)
va_m, va_s = val_sc.mean(1), val_sc.std(1)

plt.figure(figsize=(7.5, 4.5))
plt.plot(sizes, tr_m, "o-", color="#4C72B0", label="train AUC")
plt.fill_between(sizes, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color="#4C72B0")
plt.plot(sizes, va_m, "o-", color="#C44E52", label="CV (validation) AUC")
plt.fill_between(sizes, va_m - va_s, va_m + va_s, alpha=0.15, color="#C44E52")
plt.title(f"Learning curve — {champion}  (final train-CV gap = {tr_m[-1] - va_m[-1]:.3f})")
plt.xlabel("training samples"); plt.ylabel("ROC-AUC"); plt.ylim(0.6, 1.0)
plt.legend(); plt.tight_layout(); plt.show()

verdict = ("overfitting — regularise / add data" if tr_m[-1] - va_m[-1] > 0.05
           else ("underfitting — add capacity / features" if va_m[-1] < 0.7 else "well-fit"))
print(f"Train AUC={tr_m[-1]:.3f} | CV AUC={va_m[-1]:.3f} | gap={tr_m[-1]-va_m[-1]:.3f} -> {verdict}")

## 10. Feature reduction — how many of the 112 features do we really need?

112 features is wide: it raises maintenance cost, drift surface, and explainability burden. We rank features by **mean |SHAP| on the validation fold** (never on OOT), accumulate normalised importance, and plot **cumulative importance vs feature count**. We then **refit on train+val with the shortlist** and **score the sealed OOT test once** — if AUC barely moves, we ship the smaller model as the **final champion** (Section 11).

> **Reviewer:** pick the knee of the curve (or the 90/95% line), set `KEEP_FRACTION`, and either feed the shortlist into `select_feature_columns` or persist it for the trainer. Re-run after any feature-engineering change.

In [ ]:
KEEP_FRACTION = 0.95  # target cumulative |SHAP| mass to retain in the reduced model

import shap

_pre = tuned_pipe.named_steps["preprocessor"]
_model = tuned_pipe.named_steps["model"]
X_val_t = _pre.transform(X_val)

explainer = shap.TreeExplainer(_model)
shap_values = explainer.shap_values(X_val_t)
if isinstance(shap_values, list):
    shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]

# Rank by mean |SHAP| on validation only — OOT is not used for selection.
imp_full = (
    pd.Series(np.abs(shap_values).mean(axis=0), index=feature_cols)
    .sort_values(ascending=False)
)
cum = imp_full.cumsum() / max(imp_full.sum(), 1e-12)
rank = np.arange(1, len(cum) + 1)
n_90 = int((cum < 0.90).sum() + 1)
n_95 = int((cum < 0.95).sum() + 1)
n_97 = int((cum < 0.97).sum() + 1)
n_99 = int((cum < 0.99).sum() + 1)
print(
    f"Validation |SHAP| — features to reach "
    f"90%: {n_90} | 95%: {n_95} | 97%: {n_97} | 99%: {n_99} | total: {len(feature_cols)}"
)

plt.figure(figsize=(8.5, 4.5))
plt.plot(rank, cum.values, color="#4C72B0", lw=2)
for frac, n, col in [(0.90, n_90, "orange"), (0.95, n_95, "red")]:
    plt.axhline(frac, ls="--", color=col, lw=1)
    plt.axvline(n, ls=":", color=col, lw=1)
    plt.text(n + 1, 0.25, f"{n} feats", color=col)
plt.title("Cumulative validation |SHAP| importance vs feature count")
plt.xlabel("number of features (ranked by importance)")
plt.ylabel("cumulative importance")
plt.ylim(0, 1.02)
plt.tight_layout()
plt.show()

# Shortlist from validation SHAP; refit on dev; evaluate OOT blind.
reduced = imp_full.index[: int((cum < KEEP_FRACTION).sum() + 1)].tolist()
red_pipe = _build_candidates(reduced, tuned={champion: best_params_clean})[champion]
red_pipe.fit(X_dev[reduced], y_dev)
red_auc = roc_auc_score(y_oot, red_pipe.predict_proba(X_oot[reduced])[:, 1])
print(
    f"OOT AUC  full ({len(feature_cols)} feats) = {oot['roc_auc']:.4f}  |  "
    f"reduced ({len(reduced)} feats, {int(KEEP_FRACTION * 100)}% val |SHAP|) = {red_auc:.4f}  |  "
    f"delta = {red_auc - oot['roc_auc']:+.4f}"
)
print("\nShortlist (top 20):")
print(reduced[:20])

## 11. Final champion — results summary (assignment report)

The **promoted model** is the feature-reduced champion: top features covering **95% validation |SHAP|** mass, refit on train+val with tuned hyper-parameters from Section 5. This table is the headline performance for the assignment report.

| | Primary | | Secondary | | |
|---|---|---|---|---|---|
| **Split** | ROC-AUC | Gini | KS | PR-AUC | Brier |


In [ ]:
# --- Final champion: feature-reduced model (95% validation |SHAP|) ---
champion_pipe = red_pipe
champion_features = reduced

_split_frames = {
    "train": (X_train, y_train),
    "validation": (X_val, y_val),
    "test": (X_test, y_test),
    "oot": (X_oot, y_oot),
}

_rows = []
for _split_name, (_X, _y) in _split_frames.items():
    _scores = champion_pipe.predict_proba(_X[champion_features])[:, 1]
    _m = _evaluate(_split_name, _y, _scores)
    _rows.append({
        "split": _split_name,
        "n": len(_y),
        "bad_rate": float(_y.mean()),
        "roc_auc": _m["roc_auc"],
        "gini": _m["gini"],
        "ks": _m["ks"],
        "pr_auc": _m["pr_auc"],
        "brier": _m["brier"],
    })

final_metrics = pd.DataFrame(_rows).set_index("split")
_oot_final = final_metrics.loc["oot"]

print(
    f"Final champion: {champion}  |  {len(champion_features)} features "
    f"({int(KEEP_FRACTION * 100)}% validation |SHAP| mass)"
)
print("\n=== PRIMARY METRICS (ROC-AUC / Gini) ===")
print(final_metrics[["roc_auc", "gini"]].to_string(float_format=lambda x: f"{x:.4f}"))
print("\n=== SECONDARY METRICS (KS / PR-AUC / Brier) ===")
print(final_metrics[["ks", "pr_auc", "brier"]].to_string(float_format=lambda x: f"{x:.4f}"))

_auc_gap = final_metrics.loc["train", "roc_auc"] - final_metrics.loc["validation", "roc_auc"]
print(f"\nTrain - validation AUC gap: {_auc_gap:+.4f}")
print(f"OOT headline  ->  AUC={_oot_final['roc_auc']:.4f}  Gini={_oot_final['gini']:.4f}  "
      f"KS={_oot_final['ks']:.4f}  PR-AUC={_oot_final['pr_auc']:.4f}  Brier={_oot_final['brier']:.4f}")


## 12. Slide 6 — Threshold tradeoff (validation set; final champion)

Compare the **feature-reduced LightGBM champion** vs the **tuned Logistic Regression challenger** (`02c`) on the validation fold only. OOT stays sealed for monitoring (Slides 7–8).

At a chosen operating point (default **75% approval rate**), read off each model's **bad capture rate**. The LightGBM threshold at that point feeds the gold predictions table at inference.


In [ ]:
# --- Slide 6: threshold tradeoff (validation only; uses final champion_pipe) ---
LR_PARAMS_PATH = MODEL_STORE_DIR / "best_params_lr_random_dev.json"
lr_tuned = json.loads(LR_PARAMS_PATH.read_text(encoding="utf-8"))["logistic_regression"]

lgbm_model = champion_pipe
lr_model = _build_candidates(feature_cols, tuned={"logistic_regression": lr_tuned})["logistic_regression"]
lr_model.fit(X_train, y_train)

lgbm_val_probs = lgbm_model.predict_proba(X_val[champion_features])[:, 1]
lr_val_probs = lr_model.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.0, 1.01, 0.01)

def threshold_curve(probs, y_true):
    rows = []
    n_bad = y_true.sum()
    for T in thresholds:
        approved = probs < T
        approval_rate = approved.mean()
        bad_captured = ((probs >= T) & (y_true == 1)).sum() / n_bad
        rows.append({
            "threshold": T,
            "approval_rate": approval_rate,
            "bad_capture_rate": bad_captured,
        })
    return pd.DataFrame(rows)

lgbm_curve = threshold_curve(lgbm_val_probs, y_val.values)
lr_curve = threshold_curve(lr_val_probs, y_val.values)

ref_approval = 0.75

fig, ax = plt.subplots(figsize=(9, 6))
ax.grid(False)

ax.plot(
    lgbm_curve["approval_rate"], lgbm_curve["bad_capture_rate"],
    label="LightGBM (champion, reduced)", color="#2563EB", linewidth=2.5,
)
ax.plot(
    lr_curve["approval_rate"], lr_curve["bad_capture_rate"],
    label="Logistic Regression", color="#9CA3AF", linewidth=2, linestyle="--",
)

ax.axvline(
    ref_approval, color="red", linestyle=":", linewidth=1.5,
    label=f"Operating point ({ref_approval:.0%} approval)",
)

for curve, model_name, color in [
    (lgbm_curve, "LightGBM", "#2563EB"),
    (lr_curve, "LR", "#6B7280"),
]:
    row = curve.iloc[(curve["approval_rate"] - ref_approval).abs().argmin()]
    ax.annotate(
        f"{model_name}: {row['bad_capture_rate']:.1%} bads rejected",
        xy=(row["approval_rate"], row["bad_capture_rate"]),
        xytext=(row["approval_rate"] - 0.18, row["bad_capture_rate"] - 0.05),
        arrowprops=dict(arrowstyle="->", color=color),
        fontsize=10, color=color,
    )

ax.set_xlabel("Approval Rate (% of applicants approved)", fontsize=16)
ax.set_ylabel("Bad Capture Rate (% of bad customers rejected)", fontsize=16)
ax.set_title(
    "Threshold Tradeoff: Approval Rate vs Bad Capture Rate\n"
    f"(Validation set; {len(champion_features)}-feature champion)",
    fontsize=18,
)
ax.tick_params(axis="both", labelsize=13)
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()

chart_path = ROOT / "slide6_threshold_tradeoff.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved chart -> {chart_path}")

ref_row_lgbm = lgbm_curve.iloc[(lgbm_curve["approval_rate"] - ref_approval).abs().argmin()]
ref_row_lr = lr_curve.iloc[(lr_curve["approval_rate"] - ref_approval).abs().argmin()]
slide6_lgbm_threshold = float(ref_row_lgbm["threshold"])

print("=== SLIDE 6 HEADLINE NUMBERS ===")
print(f"At ~{ref_approval:.0%} approval rate:")
print(f"  LightGBM bad capture rate : {ref_row_lgbm['bad_capture_rate']:.1%}  (threshold ≈ {ref_row_lgbm['threshold']:.2f})")
print(f"  Logistic Regression       : {ref_row_lr['bad_capture_rate']:.1%}  (threshold ≈ {ref_row_lr['threshold']:.2f})")
print(f"  Delta (LightGBM advantage): {ref_row_lgbm['bad_capture_rate'] - ref_row_lr['bad_capture_rate']:+.1%}")
print(f"\nLightGBM operating threshold for inference: {slide6_lgbm_threshold:.3f}")


## 13. Model card & handoff

Headline numbers below come from the **feature-reduced champion** (Section 11), not the interim full-feature model from Section 5 holdout.

| Field | Value |
|---|---|
| **Champion family** | (printed below) |
| **Feature shortlist** | 95% validation \|SHAP\| mass |
| **Selection metric** | ROC-AUC / Gini |
| **Decision rule** | decline if `PD ≥ slide6_threshold` (from Section 12 tradeoff) |
| **Dev split** | Random train/val/test + temporal OOT (3 cohorts) |
| **Leakage controls** | PIT join, impute/scale inside pipeline, OOT split |

### Promote to production
1. Section 5 writes tuned hyper-parameters to `model_store/best_params.json`.
2. The cell below writes the **36-feature reduced champion** to `model_store/champion_config.json` (consumed by `utils/ml/train.py`).
3. Run `python -m utils.ml.train --snapshotdate 2024-12-01 --force-retrain` (or trigger the Airflow `train_model` task).
4. The DAG then back-scores history and refreshes monitoring.


In [ ]:
champion_export = {
    "source_notebook": "02b_model_tuning_random_dev_oot",
    "selected_model": champion,
    "shap_keep_fraction": KEEP_FRACTION,
    "decision_threshold": round(float(slide6_lgbm_threshold), 3),
    "best_params": best_params_clean,
    "feature_columns": champion_features,
}
MODEL_STORE_DIR.mkdir(parents=True, exist_ok=True)
champion_path = MODEL_STORE_DIR / CHAMPION_CONFIG_FILENAME
champion_path.write_text(json.dumps(champion_export, indent=2), encoding="utf-8")
print(f"Wrote champion config -> {champion_path}")

summary = {
    **champion_export,
    "n_features": len(champion_features),
    "cv_roc_auc": round(float(search.best_score_), 4),
    "oot_roc_auc": round(float(_oot_final["roc_auc"]), 4),
    "oot_gini": round(float(_oot_final["gini"]), 4),
    "oot_ks": round(float(_oot_final["ks"]), 4),
    "oot_pr_auc": round(float(_oot_final["pr_auc"]), 4),
    "oot_brier": round(float(_oot_final["brier"]), 4),
    "slide6_bad_capture_at_75pct_approval": round(float(ref_row_lgbm["bad_capture_rate"]), 4),
}
print(json.dumps(summary, indent=2))